In [6]:
from coffea.util import load
from ML.fileset import fileset
import numpy as np

In [9]:
from coffea.util import load
from ML.fileset import fileset

lum = 3000
cutflow = load("output.coffea")["nEvents"]["cutflow"]["1-lep"]

cut_order = ["1-lep", "nJet", "nBJet", "nNotBJet", "HT_jets", "MET"]
col_headers = [
    r"$n_{\ell}\geq1$",
    r"$n_{j}\geq6$",
    r"$n_{b}\geq3$",
    r"$n_{\bar{b}}\geq1$",
    r"$H_T^{j}>600$",
    r"$MET>30$",
]

def tex_escape(s):
    return s.replace("_", r"\_")

all_samples = list(cutflow.keys())
signal_samples = [s for s in all_samples if s.startswith("Signal")]
background_samples = [s for s in all_samples if not s.startswith("Signal")]

def compute_yield_rows(sample_list):
    rows = []
    for smpl in sample_list:
        weight = (lum * fileset[smpl]["metadata"]["xsec"] * 1000) / fileset[smpl]["metadata"]["nevents"]
        values = []
        for cut in cut_order:
            val = cutflow[smpl][cut] * weight
            values.append(f"{val:.2f}")
        rows.append((tex_escape(smpl), values))
    return rows

def to_latex_sci(x, precision=2):
    """Convert a float to LaTeX format. Uses plain fixed-point notation when the
    power of ten is >= 0 (x >= 1), and scientific notation only when the power
    of ten is negative (x < 1)."""
    if x == 0:
        return r"$0$"
    
    exponent = int(np.floor(np.log10(abs(x))))
    
    if exponent >= 0:
        return f"${x:.{precision}f}$"
    
    mantissa = x / (10 ** exponent)
    # Avoid cases like 10.00 x 10^-3 due to rounding
    if round(mantissa, precision) >= 10:
        mantissa /= 10
        exponent += 1
    return rf"${mantissa:.{precision}f}\times10^{{{exponent}}}$"

    
def compute_efficiency_rows(sample_list):
    """Efficiency = value / primary for each cut, expressed as percent (numbers are unitless in the table itself)."""
    rows = []
    for smpl in sample_list:
        primary = cutflow[smpl]["primary"]
        values = []
        for cut in cut_order:
            eff = (cutflow[smpl][cut] / primary) * 100
            values.append(to_latex_sci(eff))
        rows.append((tex_escape(smpl), values))
    return rows

def build_table(rows, caption, label, use_booktabs=False):
    ncols = len(cut_order)
    col_spec = "l" + "r" * ncols
    top_rule = r"\toprule" if use_booktabs else r"\hline"
    mid_rule = r"\midrule" if use_booktabs else r"\hline"
    bottom_rule = r"\bottomrule" if use_booktabs else r"\hline"
    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(r"\resizebox{\textwidth}{!}{%")
    lines.append(rf"\begin{{tabular}}{{{col_spec}}}")
    lines.append(top_rule)
    header_row = "Sample & " + " & \n\t\t\t\t".join(col_headers) + r" \\"
    lines.append(header_row)
    lines.append(mid_rule)
    for smpl, values in rows:
        row_str = smpl + " & " + " & ".join(values) + r" \\"
        lines.append(row_str)
    lines.append(bottom_rule)
    lines.append(r"\end{tabular}%")
    lines.append(r"}")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\end{table}")
    return "\n".join(lines)

USE_BOOKTABS = False  # set True if \usepackage{booktabs} is in your preamble

# --- Yield tables (as before) ---
signal_yield_rows = compute_yield_rows(signal_samples)
background_yield_rows = compute_yield_rows(background_samples)

signal_yield_table = build_table(
    signal_yield_rows,
    caption=rf"Cutflow yields for signal samples, normalized to {lum}~fb$^{{-1}}$.",
    label="tab:cutflow_yield_signal",
    use_booktabs=USE_BOOKTABS,
)
background_yield_table = build_table(
    background_yield_rows,
    caption=rf"Cutflow yields for background samples, normalized to {lum}~fb$^{{-1}}$.",
    label="tab:cutflow_yield_background",
    use_booktabs=USE_BOOKTABS,
)

# --- Efficiency tables (new) ---
signal_eff_rows = compute_efficiency_rows(signal_samples)
background_eff_rows = compute_efficiency_rows(background_samples)

signal_eff_table = build_table(
    signal_eff_rows,
    caption=r"Cumulative cut efficiencies for signal samples. All values are given in percent (\%).",
    label="tab:cutflow_eff_signal",
    use_booktabs=USE_BOOKTABS,
)
background_eff_table = build_table(
    background_eff_rows,
    caption=r"Cumulative cut efficiencies for background samples. All values are given in percent (\%).",
    label="tab:cutflow_eff_background",
    use_booktabs=USE_BOOKTABS,
)

print(signal_yield_table)
print()
print(background_yield_table)
print()
print(signal_eff_table)
print()
print(background_eff_table)

# with open("cutflow_yield_signal.tex", "w") as f:
#     f.write(signal_yield_table)
# with open("cutflow_yield_background.tex", "w") as f:
#     f.write(background_yield_table)
# with open("cutflow_eff_signal.tex", "w") as f:
#     f.write(signal_eff_table)
# with open("cutflow_eff_background.tex", "w") as f:
#     f.write(background_eff_table)

\begin{table}[htbp]
\centering
\resizebox{\textwidth}{!}{%
\begin{tabular}{lrrrrrr}
\hline
Sample & $n_{\ell}\geq1$ & 
				$n_{j}\geq6$ & 
				$n_{b}\geq3$ & 
				$n_{\bar{b}}\geq1$ & 
				$H_T^{j}>600$ & 
				$MET>30$ \\
\hline
Signal\_1500 & 933.70 & 786.82 & 232.68 & 232.64 & 230.87 & 218.00 \\
Signal\_2000 & 395.27 & 324.13 & 87.35 & 87.35 & 86.96 & 82.74 \\
Signal\_1400 & 1080.63 & 913.13 & 276.40 & 276.35 & 273.75 & 256.44 \\
Signal\_1300 & 1238.53 & 1052.60 & 322.76 & 322.73 & 319.17 & 298.91 \\
Signal\_1200 & 1392.60 & 1187.13 & 369.03 & 369.01 & 363.49 & 337.70 \\
Signal\_1100 & 1490.20 & 1272.26 & 396.54 & 396.50 & 389.07 & 359.93 \\
Signal\_1000 & 1568.24 & 1337.27 & 420.55 & 420.53 & 409.47 & 376.55 \\
Signal\_900 & 1581.59 & 1347.81 & 421.03 & 421.02 & 405.88 & 370.38 \\
Signal\_800 & 1511.58 & 1280.44 & 400.61 & 400.55 & 379.90 & 347.17 \\
Signal\_700 & 1396.28 & 1166.23 & 354.38 & 354.37 & 332.50 & 303.77 \\
Signal\_600 & 1183.99 & 1000.72 & 304.82 & 304.82 & 290.07 & 26

In [3]:
lum = 3000
cutflow = load("output.coffea")["nEvents"]["cutflow"]["1-lep"]
for smpl in cutflow:
    weight = (lum * fileset[smpl]["metadata"]["xsec"] * 1000)/fileset[smpl]["metadata"]["nevents"]
    for cut, value in cutflow[smpl].items():
        print(smpl, cut, f"{value*weight:.2f}")
        # I want to add value to a latex table that each row is sample and each column is the acumulated cut

tttJets primary 1569.00
tttJets 1-lep 420.72
tttJets nJet 353.03
tttJets nBJet 146.81
tttJets nNotBJet 146.79
tttJets HT_jets 118.68
tttJets MET 103.99
tttt primary 36840.00
tttt 1-lep 11575.50
tttt nJet 11146.16
tttt nBJet 5893.44
tttt nNotBJet 5893.15
tttt HT_jets 5566.08
tttt MET 4969.42
tt primary 3974010000.00
tt 1-lep 813859364.96
tt nJet 227663084.88
tt nBJet 21509329.12
tt nNotBJet 21509329.12
tt HT_jets 9950921.04
tt MET 8363304.05
ttH primary 1421700.00
ttH 1-lep 294483.83
ttH nJet 160908.01
ttH nBJet 75925.89
ttH nNotBJet 75878.97
ttH HT_jets 42875.63
ttH MET 36765.16
ttZ primary 2139390.00
ttZ 1-lep 460159.26
ttZ nJet 215306.07
ttZ nBJet 35494.62
ttZ nNotBJet 35488.20
ttZ HT_jets 21126.48
ttZ MET 17592.20
ttW primary 1196100.00
ttW 1-lep 321087.06
ttW nJet 100662.58
ttW nBJet 10923.98
ttW nNotBJet 10923.98
ttW HT_jets 5496.08
ttW MET 4613.36
tZJets primary 841920.00
tZJets 1-lep 217090.76
tZJets nJet 30998.65
tZJets nBJet 3038.49
tZJets nNotBJet 3037.65
tZJets HT_jets 1211.